# TITLE GOES HERE

## Setup and Imports

This section installs and imports all the necessary libraries for the project. This includes libraries for data manipulation, model training, and visualization.

In [ ]:
%pip install scikit-learn transformers torch kagglehub matplotlib seaborn pandas numpy --upgrade

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import kagglehub, os, re, random, torch, warnings, transformers
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
random.seed(42)                # Sets Python's random module seed for reproducible random operations
np.random.seed(42)             # Sets NumPy's random seed for consistent results in NumPy-based randomness
torch.manual_seed(42)          # Sets PyTorch's CPU random seed for reproducible weight initialization and shuffling
transformers.set_seed(42)      # (Optional) Sets all relevant random seeds in Hugging Face Transformers for full reproducibility
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(42) # Sets PyTorch's GPU random seed for reproducibility in CUDA (GPU) computations
  print('CUDA available:', torch.cuda.is_available())
else:
  print('CUDA not available')
# sns.set_style('darkgrid')
warnings.filterwarnings('ignore')

## Data Loading and Preprocessing

Here, we load the IMDb dataset, clean the text data by removing HTML tags and special characters, and then split it into training, validation, and testing sets.

In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))
display(df)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z ]', '', text)  # Remove special chars/digits
    text = ' '.join(text.split())  # Normalize whitespace
    return text

df['review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print('Train:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)

## BERT Model Setup

This part of the notebook sets up the BERT tokenizer and defines a custom dataset class to prepare the data for the model.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased') # alt use 'distilbert-base-uncased'

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, reviews, labels, tokenizer, max_len=128):
        self.reviews = reviews
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.reviews)
    def __getitem__(self, idx):
        text = str(self.reviews.iloc[idx])
        label = int(self.labels.iloc[idx])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = IMDbDataset(train_df['review'], train_df['label'], tokenizer)
val_dataset = IMDbDataset(val_df['review'], val_df['label'], tokenizer)
test_dataset = IMDbDataset(test_df['review'], test_df['label'], tokenizer)

## Model Training

We define the BERT model for sequence classification, set up the training arguments, and create a `Trainer` instance to fine-tune the model on our dataset.

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none'  # Disable wandb or tensorboard logging if not used
)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

train_result = trainer.train()

## Visualize Training Results

After training, we plot the training and validation loss, as well as the validation accuracy, to assess the model's learning progress.

In [ ]:
# Extract logs from Trainer's state.log_history
train_loss = []
val_loss = []
epochs = []
val_accuracy = []

for log in trainer.state.log_history:
    if 'loss' in log and 'epoch' in log and 'eval_loss' not in log:
        train_loss.append(log['loss'])
        epochs.append(log['epoch'])
    if 'eval_loss' in log:
        val_loss.append(log['eval_loss'])
        val_accuracy.append(log.get('eval_accuracy', None))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs, train_loss, label='Training Loss')
if val_loss:
    plt.plot(range(1, len(val_loss)+1), val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
if all(x is not None for x in val_accuracy):
    plt.figure(figsize=(8,5))
    plt.plot(range(1, len(val_accuracy)+1), val_accuracy, marker='o', label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy per Epoch')
    plt.legend()
    plt.show()

## Model Evaluation

The model is evaluated on the validation and test sets. We also compute and display a confusion matrix to understand the types of errors the model makes.

In [ ]:
val_results = trainer.evaluate(val_dataset)
print('Validation Results:', val_results)

In [ ]:
results = trainer.evaluate(test_dataset)
print('Test Results:', results)

In [ ]:
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

print('Accuracy:', accuracy_score(labels, preds))
print('Precision:', precision_score(labels, preds))
print('Recall:', recall_score(labels, preds))
print('F1 Score:', f1_score(labels, preds))

In [ ]:
cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(cmap='Blues')
plt.title('BERT Model - Confusion Matrix')
plt.show()

## Analyze Misclassified Examples

We examine some of the reviews that the model misclassified to gain insights into its weaknesses.

In [ ]:
misclassified = test_df.iloc[(preds != test_df['label'])]
print('Number of misclassified reviews:', len(misclassified))
print(misclassified[['review', 'sentiment']].head(10))

## Save the Model

The fine-tuned model and tokenizer are saved to disk for later use.

In [ ]:
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')

## Baseline Models

To provide context for the BERT model's performance, we train and evaluate several traditional machine learning models.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df['review'])
X_test = vectorizer.transform(test_df['review'])

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, train_df['label'])
lr_preds = lr.predict(X_test)

print('Baseline Accuracy:', accuracy_score(test_df['label'], lr_preds))
print('Baseline Precision:', precision_score(test_df['label'], lr_preds))
print('Baseline Recall:', recall_score(test_df['label'], lr_preds))
print('Baseline F1 Score:', f1_score(test_df['label'], lr_preds))

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC() # i think we use
svm.fit(X_train, train_df['label'])
svm_preds = svm.predict(X_test)

print('SVM Accuracy:', accuracy_score(test_df['label'], svm_preds))
print('SVM Precision:', precision_score(test_df['label'], svm_preds))
print('SVM Recall:', recall_score(test_df['label'], svm_preds))
print('SVM F1 Score:', f1_score(test_df['label'], svm_preds))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, train_df['label'])
nb_preds = nb.predict(X_test)

print('Naive Bayes Accuracy:', accuracy_score(test_df['label'], nb_preds))
print('Naive Bayes Precision:', precision_score(test_df['label'], nb_preds))
print('Naive Bayes Recall:', recall_score(test_df['label'], nb_preds))
print('Naive Bayes F1 Score:', f1_score(test_df['label'], nb_preds))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, train_df['label'])
rf_preds = rf.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(test_df['label'], rf_preds))
print('Random Forest Precision:', precision_score(test_df['label'], rf_preds))
print('Random Forest Recall:', recall_score(test_df['label'], rf_preds))
print('Random Forest F1 Score:', f1_score(test_df['label'], rf_preds))

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, train_df['label'])
dummy_preds = dummy.predict(X_test)

print('Dummy Classifier Accuracy:', accuracy_score(test_df['label'], dummy_preds))